In [1]:
from vertex_lite.custom import load_data,load_data_v2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import glob
import os
import numpy as np

In [2]:
def filter_data(local=False, cluster=False):
    if cluster:
        rpath = 'resultados_cluster'
    else:
        rpath = 'resultados'

    dfs = []
    for pct in [0.05, 0.25]:
        dfs.append(load_data(
            f'{rpath}/noise_*/pct_{pct}/sim_*/cells.parquet',
            columns=['time', 'n_lados', 'neighbor_of_mutant', 'cell_id', 'alive']
        ))

    df = pd.concat(dfs, ignore_index=True)

    if df.empty:
        print("No data found")
        return pd.DataFrame()

    df = df[df['alive'] == 1]

    if local:
        df = df[df['neighbor_of_mutant'] == True]
    else:
        df = df.drop(columns=['neighbor_of_mutant'])

    df = (
        df[df['n_lados'] >= 3]
        .groupby(['time', 'mesh_type', 'pct_mutant', 'sim_id', 'n_lados'])
        .size()
        .reset_index(name='n_cells')
    )

    return df


In [3]:
df=filter_data()

In [4]:
def compute_hex_deviation(df):
    def mad_hex(group):
        N = group["n_cells"].sum()
        return (group["n_cells"] * (group["n_lados"] - 6).abs()).sum() / N

    dev = (
        df.groupby(["time", "mesh_type", "pct_mutant", "sim_id"])
        .apply(mad_hex,include_groups=False)
        .reset_index(name="deviation")
    )

    return dev

import numpy as np
def compute_hex_deviation2(df):
    df2 = df.copy()
    df2["abs_dev_hex"] = (df2["n_lados"] - 6).abs() * df2["n_cells"]

    dev = (
        df2.groupby(["time", "mesh_type", "pct_mutant", "sim_id"])
        .agg(
            total_abs_dev=("abs_dev_hex", "sum"),
            total_cells=("n_cells", "sum")
        )
        .reset_index()
    )

    dev["deviation"] = dev["total_abs_dev"] / dev["total_cells"]

    return dev[["time", "mesh_type", "pct_mutant", "sim_id", "deviation"]]
def summarize_hex_deviation(dev):
    stats = (
        dev.groupby(["time", "mesh_type", "pct_mutant"])["deviation"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    stats["sem"] = stats["std"] / np.sqrt(stats["count"])
    return stats


In [5]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors
from matplotlib.cm import ScalarMappable

def plot_hex_deviation(stats, local=False, cluster=False,max_time=5000):
    sns.set_theme(
        style="white",
        context="paper",
        rc={
            "axes.edgecolor": "black",
            "axes.linewidth": 1.0,
            "axes.facecolor": "white",
            "figure.facecolor": "white",
        }
    )

    os.makedirs("figures_paper/hex_time_2", exist_ok=True)

    local_label = "local" if local else "global"
    cluster_label = "cluster" if cluster else "uniform"

    for pct in sorted(stats["pct_mutant"].unique()):
        d0 = stats[stats["pct_mutant"] == pct].copy()
        if d0.empty:
            continue

        fig, ax = plt.subplots(figsize=(6.5, 4))

        meshes_sorted = sorted(d0["mesh_type"].unique())
        norm = colors.Normalize(vmin=min(meshes_sorted), vmax=max(meshes_sorted))
        cmap = sns.light_palette("#2B352A", as_cmap=True)
        # cmap = sns.color_palette("ch:s=-.2,r=.1", as_cmap=True)
        

        for mesh in meshes_sorted:
            d = d0[d0["mesh_type"] == mesh].sort_values("time")
            d = d[d["time"] <= max_time]

            x = d["time"].to_numpy()
            y = d["mean"].to_numpy()
            sem = d["sem"].to_numpy()
            color = cmap(norm(mesh))

            ax.fill_between(
                x, y - sem, y + sem,
                color=color, alpha=0.18, linewidth=0
            )
            ax.plot(
                x, y,
                color=color, linewidth=2.25
            )

        ax.set_title(
            f"Mutant percentage {pct} | {local_label} | {cluster_label}",
            fontsize=14
        )
        ax.set_xlabel("Time step", fontsize=16)
        ax.set_ylabel("Mean absolute deviation from hexagon", fontsize=16)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(
            axis='both', which='both', direction='out',
            length=4, labelsize=13, width=1
        )

        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.05)
        cbar.set_label("Initial mesh noise", fontsize=14)
        cbar.ax.tick_params(labelsize=12)
        cbar.outline.set_visible(False)
        plt.ylim([0, 1.3])
        plt.tight_layout()
        fig.savefig(
            f"figures_paper/hex_time_2/hex_deviation_pct_{pct}_{local_label}_{cluster_label}_{max_time}.svg",
            format="svg"
        )
        plt.close(fig)

In [6]:

for local in [True,False]:
    for cluster in [True,False]:

        df = filter_data(local=local, cluster=cluster)
        dev = compute_hex_deviation2(df)
        stats = summarize_hex_deviation(dev)
        plot_hex_deviation(stats, local=local, cluster=cluster, max_time= 15000 if cluster else 2000)

In [7]:
# for local in [True,False]:
#     for cluster in [True,False]:

#         # df = filter_data(local=local, cluster=cluster)
#         # dev = compute_hex_deviation2(df)
#         # stats = summarize_hex_deviation(dev)
#         plot_hex_deviation(stats, local=local, cluster=cluster)